# mPES: optimizacion y entrenamiento en Google Colab

Este notebook ejecuta trabajos largos de `h1`, `h2` o `h3` usando el almacenamiento local rapido de Colab y Google Drive como bucket persistente.

1. Configurar Colab y seleccionar GPU/TPU.
2. Instalar y validar dependencias.
3. Montar Drive y definir linea, paquetes y operacion.
4. Preparar y sincronizar artefactos.
5. Ejecutar optimizaciones bayesianas o entrenamientos.
6. Revisar progreso y resultados.
7. Exportar artefactos para reanudar o usar localmente.

> Este notebook no ejecuta benchmarks, `general/orchestrate.py`, agregaciones ni graficos comparativos.

## 1. Configurar el entorno de Google Colab

En Colab selecciona **Entorno de ejecucion > Cambiar tipo de entorno de ejecucion** y elige GPU para DQN, RDQN, TRF y A2C. Los modelos tabulares pueden ejecutarse en CPU. TPU no es necesaria para este runner.

In [ ]:
"""Prepare the Colab runtime: clone/update the mPES repo and detect hardware."""
import importlib
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

REPO_URL = 'https://github.com/Maximiliano0/mPES.git'
REPO_REF = 'new_uq'
REPO_ROOT = Path('/content/mPES')

if not (REPO_ROOT / '.git').is_dir():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
        print('Eliminada una clonacion incompleta:', REPO_ROOT)
    clone_error = None
    for attempt in range(1, 4):
        result = subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_ROOT)],
            text=True,
            capture_output=True,
            check=False,
        )
        if result.returncode == 0:
            break
        clone_error = result.stderr.strip() or result.stdout.strip()
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        if attempt < 3:
            print(f'Clonacion fallida (intento {attempt}/3); reintentando...')
            time.sleep(3)
    else:
        raise RuntimeError(
            f'No se pudo clonar {REPO_URL} en tres intentos. Error de Git:\n{clone_error}')
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard', f'origin/{REPO_REF}'], check=True)

print('Repositorio del proyecto:', REPO_URL)
print('Rama:', REPO_REF)
print('Python:', sys.version)
print('Sistema:', platform.platform())
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
except (FileNotFoundError, subprocess.CalledProcessError):
    print('GPU NVIDIA no detectada')
try:
    import tensorflow as tf
    print('TensorFlow:', tf.__version__)
    print('GPUs:', tf.config.list_physical_devices('GPU'))
    print('TPUs:', tf.config.list_logical_devices('TPU'))
except ImportError:
    print('TensorFlow aun no esta instalado')

drive = importlib.import_module('google.colab.drive')
drive.mount('/content/drive')
assert (REPO_ROOT / 'h1').is_dir(), f'Falta h1 en {REPO_ROOT}'
assert (REPO_ROOT / 'h2').is_dir(), f'Falta h2 en {REPO_ROOT}'
assert (REPO_ROOT / 'h3').is_dir(), f'Falta h3 en {REPO_ROOT}'
assert (REPO_ROOT / 'utils' / 'config' / 'requirements.txt').is_file(), 'Falta requirements.txt'
print('Repositorio validado:', REPO_ROOT)

## 2. Instalar y validar dependencias

Ejecuta la instalacion una vez por sesion. Si Colab solicita reiniciar el entorno, reinicialo y vuelve a ejecutar las celdas desde el principio.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/mPES/utils/config/requirements.txt'],
    check=True,
)

In [ ]:
metadata_module = importlib.import_module('importlib.metadata')
requirements_module = importlib.import_module('packaging.requirements')
Requirement = requirements_module.Requirement

requirements_file = Path(REPO_ROOT) / 'utils' / 'config' / 'requirements.txt'
assert requirements_file.is_file(), f'No existe: {requirements_file}'

mismatches = []
checked = []
for raw_line in requirements_file.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    requirement = Requirement(line)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    package_name = requirement.name
    try:
        installed_version = metadata_module.version(package_name)
    except metadata_module.PackageNotFoundError:
        mismatches.append(f'{package_name}: no instalado')
        continue
    checked.append(package_name)
    if requirement.specifier and installed_version not in requirement.specifier:
        mismatches.append(f'{package_name}: {installed_version} != {requirement.specifier}')

assert not mismatches, 'Versiones incompatibles:\n' + '\n'.join(mismatches)
print(f'Requirements compatibles: {len(checked)} paquetes verificados')

## 3. Definir rutas y parámetros de trabajo

Edita solo las variables de la siguiente celda. `DRIVE_ROOT` es el bucket persistente; el runner trabaja sobre `REPO_ROOT` en el disco local de Colab. Usa `h1`, `h2` o `h3` de forma explícita.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/mPES-bucket'
LINE = 'h1'                 # 'h1', 'h2' o 'h3'
PACKAGES = 'pes_dqn,pes_rdqn,pes_trf'
OPERATION = 'optimize'      # 'optimize' o 'train'
TRIALS = 30
RESUME_DATE = None           # Ejemplo: '2026-04-29'
RUN_ID = f'{LINE}_{OPERATION}_colab_001'
SYNC_INTERVAL = 300

os.environ['MPES_COLAB'] = '1'
os.environ['MPES_DRIVE_ROOT'] = DRIVE_ROOT
assert REPO_ROOT.is_dir(), f'Repositorio no encontrado: {REPO_ROOT}'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Configuracion lista:', LINE, PACKAGES, OPERATION)

In [ ]:
line_root = Path(REPO_ROOT) / LINE
assert line_root.is_dir(), f'Linea no encontrada: {line_root}'
for package in [item.strip() for item in PACKAGES.split(',')]:
    matches = list(line_root.glob(f'**/{package}'))
    assert matches, f'Paquete no encontrado en {LINE}: {package}'
    print(package, '->', matches[0])

## 4. Preparar datos y sincronizar desde Drive

El runner descarga `inputs/` y `outputs/` del paquete antes de ejecutarlo. Esto permite recuperar `best_params.json`, estudios Optuna, modelos y CSV compartidos. No copies el repositorio completo sobre Drive durante una ejecucion.

In [ ]:
runner = Path(REPO_ROOT) / 'h1' / 'colab' / 'runner.py'
assert runner.is_file(), f'Runner no encontrado: {runner}'
print('Bucket:', DRIVE_ROOT)
print('Runner:', runner)
print('No se ejecutaran benchmarks:', True)

## 5. Pruebas rapidas

Estas comprobaciones no entrenan modelos: validan rutas, paquetes, operacion y la prohibicion de benchmarks antes de consumir GPU.

In [ ]:
assert LINE in {'h1', 'h2', 'h3'}
assert OPERATION in {'optimize', 'train'}
assert PACKAGES.strip()
assert SYNC_INTERVAL >= 10
print('Pruebas rapidas correctas')

## 6. Ejecutar optimizacion o entrenamiento

Selecciona `OPERATION = 'optimize'` para Optuna o `OPERATION = 'train'` para reentrenar con los mejores parametros. Ejecuta esta celda solo cuando las pruebas rapidas hayan pasado. El proceso es secuencial y sincroniza el estado a Drive cada `SYNC_INTERVAL` segundos.

In [ ]:
command = [
    sys.executable, str(runner),
    '--line', LINE,
    '--packages', PACKAGES,
    '--operation', OPERATION,
    '--trials', str(TRIALS),
    '--repo-root', REPO_ROOT,
    '--drive-root', DRIVE_ROOT,
    '--run-id', RUN_ID,
    '--sync-interval', str(SYNC_INTERVAL),
]
if RESUME_DATE:
    command.extend(['--resume-date', RESUME_DATE])
print('Ejecutando:', ' '.join(command))
subprocess.run(command, check=True, cwd=REPO_ROOT)

## 7. Visualizar y revisar el progreso

El archivo de estado vive en `Drive/runs/<RUN_ID>/`. Confirma que el paquete avance, que `returncode` sea cero y que `last_sync` se actualice. Si Colab se desconecta, vuelve a montar Drive y ejecuta el mismo `RUN_ID` con `RESUME_DATE` cuando el optimizador lo soporte.

In [ ]:
import json
from IPython.display import display, JSON

status_files = sorted((Path(DRIVE_ROOT) / 'runs' / RUN_ID).glob('**/status.json'))
assert status_files, 'Aun no hay status.json; ejecuta primero la celda de lanzamiento.'
for status_file in status_files:
    print(status_file.relative_to(Path(DRIVE_ROOT)))
    display(JSON(json.loads(status_file.read_text(encoding='utf-8'))))

## 8. Guardar artefactos y exportar resultados

El runner sincroniza automaticamente `inputs/`, `outputs/`, logs, `manifest.json` y `status.json`. Para reproducir desde cero: monta Drive, usa el mismo `DRIVE_ROOT`, selecciona la misma linea y paquete, recupera el mismo `RUN_ID` y ejecuta con `RESUME_DATE` si corresponde.

In [ ]:
export_path = Path('/content') / f'{RUN_ID}.zip'
source_path = Path(DRIVE_ROOT) / 'runs' / RUN_ID
assert source_path.is_dir(), f'No existe la ejecucion en Drive: {source_path}'
shutil.make_archive(str(export_path.with_suffix('')), 'zip', source_path)
print('Archivo exportado:', export_path)
try:
    files_module = importlib.import_module('google.colab.files')
    files_module.download(str(export_path))
except ImportError:
    print('Descarga manual:', export_path)